<a href="https://colab.research.google.com/github/ujwaljain506-hash/anime-recommender/blob/main/day3_recommendation_function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/anime-dataset-2023.csv')
print("File Loaded")

File Loaded


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Genres'])

print(tfidf_matrix.shape)

(24905, 28)


In [7]:
print(tfidf.get_feature_names_out())

['action' 'adventure' 'avant' 'award' 'boys' 'comedy' 'drama' 'ecchi'
 'erotica' 'fantasy' 'fi' 'garde' 'girls' 'gourmet' 'hentai' 'horror'
 'life' 'love' 'mystery' 'of' 'romance' 'sci' 'slice' 'sports'
 'supernatural' 'suspense' 'unknown' 'winning']


In [8]:
df['Genres'] = df['Genres'].str.replace(' ,', ',').str.replace(', ',',').str.replace(' ', '_').str.replace('-', '_')

In [9]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Genres'])
print(tfidf.get_feature_names_out())

['action' 'adventure' 'avant_garde' 'award_winning' 'boys_love' 'comedy'
 'drama' 'ecchi' 'erotica' 'fantasy' 'girls_love' 'gourmet' 'hentai'
 'horror' 'mystery' 'romance' 'sci_fi' 'slice_of_life' 'sports'
 'supernatural' 'suspense' 'unknown']


In [10]:
df = df[df['Genres'] != 'unknown']
print(df.shape)

(24905, 24)


In [11]:
df[df['Genres'].str.contains('unknown')].shape

(0, 24)

In [12]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['Genres'])
print(tfidf_matrix.shape)

(24905, 22)


Concept: Cosine Similarity
Remember our TF-IDF matrix — each anime is represented as a list of 22 numbers (one per genre). For example:

Cowboy Bebop → [0.6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.8, 0, 0, 0, 0, 0]

Trigun → [0.5, 0.7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.6, 0, 0, 0, 0, 0]

Cosine similarity measures the angle between two such lists. If two anime point in the same direction (similar genres), the angle is small → similarity score close to 1. If they're completely different, angle is large → similarity score close to 0.

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(cosine_sim.shape)

(24905, 24905)


In [14]:
print(cosine_sim[0])

[1.         0.58606646 0.47922062 ... 0.42675257 0.         0.        ]


In [20]:
def recommend_anime(title):
    # Find the index of the anime in the dataframe
    match = df[(df['Name'] == title) | (df['English name'] == title)]

    if match.empty:
        return "Anime not found! Check the spelling."

    idx = match.index[0]
    # Get similarity scores for this anime against all others
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score, highest first
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Skip the first result (the anime itself), take next 10
    sim_scores = sim_scores[1:11]

    # Get the anime indices
    anime_indices = [i[0] for i in sim_scores]

    # Return the anime names
    return df['Name'].iloc[anime_indices].values

print(recommend_anime('High School DxD'))

['Love Hina' 'Ichigo 100%' 'Love♥Love?' 'Futari Ecchi'
 'Hanaukyou Maid-tai' 'Iketeru Futari' 'Ichigo 100% Special 2'
 'Hanaukyou Maid-tai: La Verite' 'Hanaukyou Maid-tai OVA'
 'Guardian Hearts']
